# NB_SLV_PN_HOTEL_RATE_OFFER

RECOMPUTE `lh_vv_silver.dbo.slv_pn_hotel_rate_offer`. Một dòng mỗi `rate_plan_id`. `valid_from` và `valid_to` chuyển nguyên giá trị.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
FLAG = "_is_deleted"
OUT_COLS = ("rate_plan_id", "status", "meal_plan", "valid_from", "valid_to")
RATE_TYPES = {
    "id": "string", "status": "string", "meal_plan": "int",
    "valid_from": "timestamp", "valid_to": "timestamp", "deleted": "boolean",
}

def _require_types(df, expected, name):
    """Chặn khi build thiếu cột hoặc kiểu lệch Field Mapping L1. Không cast.

    Input : df — DataFrame L1; expected — {cột: simpleString}; name — tên bảng
    Output: None
    Ví dụ : _require_types(products, PRODUCT_TYPES, "slv_pn_products")
    """
    got = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    missing = [col for col in expected if col not in got]
    bad = [f"{col}: {got[col]}" for col in expected if col in got and got[col] != expected[col]]
    if missing or bad:
        raise ConfigError(f"{name}: thiếu {missing}; lệch kiểu {bad}")


def _spec():
    return {
        "trg_tbl": "slv_pn_hotel_rate_offer",
        "inputs": {"rate_plans": "slv_pn_hotel_rate_plans"},
        "required_inputs": ("rate_plans",),
        "block_checks": (),
        "pk": ("rate_plan_id",),
        "deleted_flag": "_is_deleted",
        "write_mode": "MERGE",
    }


def build(ctx):
    plans = ctx.input("rate_plans")
    _require_types(plans, RATE_TYPES, "slv_pn_hotel_rate_plans")
    rows = plans.select(
        F.col("id").alias("rate_plan_id"),
        "status",
        "meal_plan",
        "valid_from",
        "valid_to",
        F.col("deleted").alias(FLAG),
    )
    return rows.select(*OUT_COLS, FLAG)


SPEC = NodeSpec(
    trg_tbl="slv_pn_hotel_rate_offer",
    inputs={"rate_plans": "slv_pn_hotel_rate_plans"},
    pk=("rate_plan_id",),
    build=build,
    required_inputs=("rate_plans",),
    deleted_flag=FLAG,
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
